# Hof_popsynth_analysis.ipynb

Data analysis of **population-synthesis sweeps** run with a `run_popsynth*.sh` script
(which launches `run_model_Hof.py` once per parameter combination).

**Generic by design (works for future sweeps with other variables / dimensionality).**
Nothing about *which* parameters were swept is hard-coded here. Everything is recovered
from the two files that actually defined the sweep:

1. **The sweep script `<name>.sh`** is parsed for the grid: every list fed to GNU
   `parallel ... ::: $X_VALUES ::: ...` (or, failing that, the fallback `for v in $X; do`
   loops) and which `run_model_Hof.py --flag "$var"` each list ends up in. Also read
   from it: `CONFIG_FILE`, `OUTDIR` (where the .h5 files are) and `LOGDIR`.
2. **`run_model_Hof.py`** is used for the CLI-flag → config-key mapping (parsed from
   its `overrides = {...}` block and its `config['sec']['key'] = args.flag` lines), for
   `output_filename()` (the single source of truth for filenames), and for
   `make_time_grid()` (the expected final snapshot time).

So a new sweep needs no notebook edits beyond `SWEEP_SCRIPT` below, as long as it keeps
the same pattern: `*_VALUES` lists → `parallel :::` / `for` loops → `--flag "$var"`. A
new `run_model_Hof.py` CLI flag is picked up automatically if it is applied with one of
those two override patterns.

`<name>` throughout = the sweep script's filename without `.sh`
(e.g. `run_popsynth1.00_hof`).

# Imports

In [1]:
# Imports
import os, re, sys, copy, json, itertools
from datetime import datetime

import numpy as np
import pandas as pd
import h5py

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)

# Settings

Change `SWEEP_SCRIPT` to analyse a different sweep. The `*_OVERRIDE` switches are only
needed when the paths written in the `.sh` file are not valid on the machine running this
notebook (e.g. working locally on files copied from Nibi); leave them `None` on Nibi.

In [2]:
# ---- which sweep -------------------------------------------------------------------
SWEEP_SCRIPT = "run_popsynth1.00_hof.sh"     # filename of the sweep's .sh launcher (in STARTHERE_DIR)

# ---- where things are --------------------------------------------------------------
# This notebook lives in example/StartHere/notebooks/; the .sh scripts, run_model_Hof.py
# and config/ live one level up. Jupyter's working directory is the notebook's folder.
NOTEBOOK_DIR  = os.path.abspath(os.getcwd())
STARTHERE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
RUN_MODEL     = os.path.join(STARTHERE_DIR, "run_model_Hof.py")   # the model the sweep ran

SWEEP_NAME = os.path.splitext(SWEEP_SCRIPT)[0]   # <name>, e.g. "run_popsynth1.00_hof"

# Optional overrides of the paths read from the .sh file (None = use the .sh value):
OUTDIR_OVERRIDE  = None   # folder holding the sweep's .h5 output files
LOGDIR_OVERRIDE  = None   # parent logs folder; the status CSV goes in <LOGDIR>/<name>/
CONFIG_OVERRIDE  = None   # the sweep's JSON config

# Optional manual grid, bypassing the .sh parser entirely, e.g.
#   {"psi_DW": [0, 10], "e_rad": [0.8, 0.9]}     keys = run_model_Hof.py CLI flag names
# Only needed if a future .sh no longer follows the parsable pattern described above.
SWEEP_GRID_MANUAL = None

# A run counts as having reached the end if its last snapshot time is within this
# RELATIVE tolerance of the last time in the config's time grid (dimensionless).
T_END_RTOL = 1e-3

# run_model_Hof.py (and its disc_setup_Hof import) must be importable for
# output_filename() / make_time_grid():
if STARTHERE_DIR not in sys.path:
    sys.path.insert(0, STARTHERE_DIR)
import run_model_Hof as rmh
from DiscEvolution.constants import yr   # code-time units per year (= 2*pi), see CLAUDE.md

# Helper functions: sweep-script parser, CLI-flag mapping, run status

All the "generic" logic. Read top to bottom:

* `parse_sweep_script()` turns `<name>.sh` into `{flag: [values]}` (+ paths).
* `cli_flag_map()` turns `run_model_Hof.py`'s source into `{flag: (section, key, type)}`.
* `config_for_run()` / `expected_filename()` rebuild one run's config and filename exactly
  as `run_model_Hof.py` did when the sweep launched it.
* `run_status()` opens one .h5 file and classifies it.

**Placeholder `none`:** the sweep scripts use the value `none` to mean "do not pass this
flag at all" (e.g. `ALPHA_SS_LOOP="${ALPHA_SS_VALUES:-none}"`). Such a value is dropped,
and a dimension whose ONLY value is `none` is not a sweep dimension (no CSV column).

In [3]:
# ---------------------------------------------------------------------------
# 1. A minimal bash reader: just enough of bash to recover the sweep grid.
# ---------------------------------------------------------------------------
_BASH_VAR = re.compile(r"\$\{(\w+)(?::-([^}]*))?\}|\$(\w+)")   # ${X}, ${X:-default}, $X

def bash_expand(s, variables):
    """Expand $X / ${X} / ${X:-default} in s using `variables` (unknown -> left as is)."""
    def repl(m):
        name = m.group(1) or m.group(3)
        default = m.group(2)
        val = variables.get(name)
        if val in (None, "") and default is not None:   # ${X:-default}: empty OR unset -> default
            return default
        return val if val is not None else m.group(0)
    return _BASH_VAR.sub(repl, s)


def _strip_comment(line):
    """Drop a full-line or trailing ' # ...' bash comment (no '#' appears inside our quoted values)."""
    if line.lstrip().startswith("#"):
        return ""
    return re.sub(r"\s+#.*$", "", line)


def read_bash_lines(path):
    """Return the script's lines with comments stripped and '\\'-continuations joined."""
    with open(path) as f:
        raw = f.read()
    raw = re.sub(r"\\\n", " ", raw)                     # join "...\<newline>..." continuations
    return [_strip_comment(l) for l in raw.splitlines()]


def bash_toplevel_vars(lines):
    """
    Top-level (UNindented) assignments NAME=value / NAME="value" / NAME='value', in
    script order, each expanded with the variables defined before it. Command
    substitutions ($(...)) are skipped (we never need them, and cannot run them).
    """
    variables = dict(os.environ)    # so e.g. $HOME resolves like it would in bash
    for line in lines:
        m = re.match(r"^([A-Za-z_]\w*)=(.*)$", line)
        if not m:
            continue
        name, rhs = m.group(1), m.group(2).strip()
        if rhs.startswith("$("):
            continue
        if rhs[:1] in ('"', "'"):
            q = rhs[0]
            val = rhs[1:rhs.index(q, 1)] if q in rhs[1:] else rhs[1:]
            if q == '"':
                val = bash_expand(val, variables)       # single quotes: no expansion, as in bash
        else:
            val = bash_expand(rhs.split()[0] if rhs else "", variables)
        variables[name] = val
    return variables


def _function_body(lines, fname):
    """Lines of `fname() { ... }` (up to the first line that is just '}'), or [] if absent."""
    for i, line in enumerate(lines):
        if re.match(rf"^\s*{re.escape(fname)}\s*\(\)\s*\{{", line):
            body = []
            for l in lines[i + 1:]:
                if l.strip() == "}":
                    return body
                body.append(l)
    return []


# ---------------------------------------------------------------------------
# 2. Sweep grid from the .sh file
# ---------------------------------------------------------------------------
def parse_sweep_script(path):
    """
    Parse a run_popsynth*.sh launcher.

    Returns dict with
        variables : all top-level bash variables (expanded strings)
        grid      : {cli_flag: [value strings]} in launch (argument) order, 'none'
                    placeholders removed, dimensions with no real value dropped
        config_file, outdir, logdir : paths as written in the script (may be None)

    How a sweep dimension is recognised (no parameter names are hard-coded):
      * positional lists: `parallel ... run_one {1} {2} ... ::: $A ::: $B ...` -> $1 = A, $2 = B, ...
        (fallback when there is no parallel line: `for v in $A; do` loops + the
         `run_one "$v1" "$v2" ...` call inside them)
      * names: `local x="$1" y="$2"` inside run_one()
      * flags: any `--flag "$x"` inside run_one() (incl. arrays like `args=(--flag "$x")`)
    """
    lines = read_bash_lines(path)
    variables = bash_toplevel_vars(lines)
    split = lambda s: bash_expand(s, variables).split()

    # -- positional argument lists: GNU parallel line first ...
    positional = None                      # list (one per $1, $2, ...) of value-string lists
    for line in lines:
        if re.search(r"\bparallel\b", line) and ":::" in line:
            positional = [split(chunk) for chunk in line.split(":::")[1:]]
            break

    body = _function_body(lines, "run_one")
    locals_ = {}                           # local name -> its values (value-string list)

    if positional is not None:
        for line in body:
            if re.match(r"^\s*local\b", line):
                for name, pos in re.findall(r'(\w+)="?\$\{?(\d+)\}?"?', line):
                    if int(pos) - 1 < len(positional):
                        locals_[name] = positional[int(pos) - 1]
    else:
        # -- ... else the plain for-loop fallback
        loop_vals = {}
        for line in lines:
            for var, lst in re.findall(r"\bfor\s+(\w+)\s+in\s+(.+?);\s*do", line):
                loop_vals[var] = split(lst)
        call = next((l for l in lines if re.match(r'^\s*run_one\s+"', l)), None)
        if call is not None and body:
            call_args = re.findall(r'"\$\{?(\w+)\}?"', call)      # loop vars, in $1, $2, ... order
            positional = [loop_vals.get(v, []) for v in call_args]
            for line in body:
                if re.match(r"^\s*local\b", line):
                    for name, pos in re.findall(r'(\w+)="?\$\{?(\d+)\}?"?', line):
                        if int(pos) - 1 < len(positional):
                            locals_[name] = positional[int(pos) - 1]
        else:
            locals_ = loop_vals           # python called directly inside the loops: flags use loop vars

    # keep locals_ in positional order ($1 first), whatever order the `local` lines list them in
    pos_of = lambda vals: next((i for i, p in enumerate(positional or []) if p is vals), len(locals_))
    locals_ = dict(sorted(locals_.items(), key=lambda kv: pos_of(kv[1])))

    # -- flags: --flag "$var" where var is one of the swept names
    found = {}                                  # flag -> swept variable name
    for line in (body or lines):
        for flag, var in re.findall(r'--(\w+)\s+"?\$\{?(\w+)\}?"?', line):
            if var in locals_ and flag not in found:
                found[flag] = var
    # order the dimensions by launch-argument order ($1, $2, ... / loop nesting order),
    # not by where each --flag happens to appear in the script
    order = list(locals_)
    grid = {}
    for flag in sorted(found, key=lambda fl: order.index(found[fl])):
        vals = [v for v in locals_[found[flag]] if v.lower() != "none"]
        if vals:
            grid[flag] = vals

    return dict(variables=variables, grid=grid,
                config_file=variables.get("CONFIG_FILE"),
                outdir=variables.get("OUTDIR"),
                logdir=variables.get("LOGDIR"))


# ---------------------------------------------------------------------------
# 3. CLI flag -> config key, read from run_model_Hof.py's own source
# ---------------------------------------------------------------------------
_ARGPARSE_TYPES = {"float": float, "int": int, "str": str}

def cli_flag_map(run_model_path):
    """
    {flag: (section, key, python_type)} for every run_model_Hof.py flag that overrides
    a config entry. Recognised source patterns (the two run_model_Hof.py uses):
        ("section", "key"): args.flag,                   (the `overrides` dict)
        config['section']['key'] = args.flag             (e.g. --alpha_SS)
    python_type comes from that flag's `add_argument(..., type=...)` (default float).
    """
    src = open(run_model_path).read()
    types = {f: _ARGPARSE_TYPES.get(t, float)
             for f, t in re.findall(r'add_argument\(\s*"--(\w+)"\s*,\s*type\s*=\s*(\w+)', src)}
    mapping = {}
    for sec, key, flag in re.findall(r'\(\s*"(\w+)"\s*,\s*"(\w+)"\s*\)\s*:\s*args\.(\w+)', src):
        mapping[flag] = (sec, key, types.get(flag, float))
    for sec, key, flag in re.findall(r"config\[['\"](\w+)['\"]\]\[['\"](\w+)['\"]\]\s*=\s*args\.(\w+)", src):
        mapping.setdefault(flag, (sec, key, types.get(flag, float)))
    return mapping


def config_for_run(base_config, params, flag_map):
    """Deep copy of the sweep config with this run's CLI overrides applied, as run_model_Hof.py does."""
    cfg = copy.deepcopy(base_config)
    for flag, value in params.items():
        sec, key, _ = flag_map[flag]
        cfg.setdefault(sec, {})[key] = value
    return cfg


def expected_filename(cfg):
    """Basename of this run's .h5 output: run_model_Hof.output_filename() is the single source of truth."""
    return rmh.output_filename(cfg)


def expected_t_end_Myr(cfg):
    """Last checkpoint time [Myr] the run should reach (make_time_grid() returns code-time units)."""
    return float(np.max(rmh.make_time_grid(cfg["simulation"]))) / (1e6 * yr)


# ---------------------------------------------------------------------------
# 4. Status of one run
# ---------------------------------------------------------------------------
# Status values (CSV column "status"):
#   complete           finished normally and its last snapshot reached the config's final time
#   Mdot_out_of_range  not evolved: initial Mdot outside the valid range (flagged file,
#                      attr Mdot_out_of_range = True, no datasets) -- see run_model_Hof.py
#   crashed            attr complete = True BUT the last snapshot is short of the final time.
#                      NOTE: run_model_Hof.py sets complete = True in a `finally:` block, so an
#                      exception inside _integrate() also leaves complete = True -- the
#                      snapshot-time check is what tells the two apart.
#   incomplete         file exists but attr complete is missing/False: killed before the
#                      `finally:` ran (SLURM time limit, node failure, ...) or still running
#   missing            no output file (never started, or killed before creating it)
#   unreadable         file exists but could not be opened (corrupt, or locked by a writer)
def run_status(path, t_end_Myr, rtol=T_END_RTOL):
    """Return (status, runtime_hr) for the .h5 file at `path`. runtime_hr = run_end - run_start [hours] or NaN."""
    if not os.path.exists(path):
        return "missing", np.nan
    try:
        try:
            f = h5py.File(path, "r", locking=False)   # don't fail on a file a live run holds open
        except TypeError:                              # old h5py without `locking`
            f = h5py.File(path, "r")
    except Exception:
        return "unreadable", np.nan

    with f:
        a = f.attrs
        runtime_hr = np.nan
        if "run_start" in a and "run_end" in a:
            # ISO-8601 local wall-clock times to the second (written by run_model_Hof.py)
            start = datetime.fromisoformat(str(a["run_start"]))
            end   = datetime.fromisoformat(str(a["run_end"]))
            runtime_hr = (end - start).total_seconds() / 3600.0          # [hours]

        if not bool(a.get("complete", False)):
            return "incomplete", runtime_hr
        if bool(a.get("Mdot_out_of_range", False)):
            return "Mdot_out_of_range", runtime_hr
        t_last = float(f["time_snap"][-1]) if ("time_snap" in f and len(f["time_snap"])) else -np.inf  # [Myr]
        if t_last < t_end_Myr * (1.0 - rtol):
            return "crashed", runtime_hr
        return "complete", runtime_hr

# Sweep status table → `<name>_status.csv`

One row per parameter combination in the sweep (the full Cartesian product of the
grid, in the order the `.sh` launched them), **including runs with no output file**.
Columns, in order:

| column | meaning |
|---|---|
| `filename` | expected .h5 basename (from `run_model_Hof.output_filename()`) |
| one column per swept CLI flag | that run's value, named after the `run_model_Hof.py` flag (e.g. `psi_DW` [dimensionless], `alpha_SS` [dimensionless], `M` [Msun], `Rd` [AU], `e_rad` [dimensionless]) |
| `status` | see `run_status()` above |
| `runtime_hr` | wall-clock `run_end - run_start` [hours], incl. calibration/setup; blank when not recorded |

Saved to `<LOGDIR>/<name>/<name>_status.csv` (`LOGDIR` from the `.sh`; the subfolder is
created if needed). Re-running the cell overwrites it with the current status.

In [4]:
# ---- 1. Parse the sweep script and resolve paths ------------------------------------
SWEEP_SCRIPT_PATH = os.path.join(STARTHERE_DIR, SWEEP_SCRIPT)
sweep = parse_sweep_script(SWEEP_SCRIPT_PATH)

def _local_fallback(p):
    """The .sh paths are Nibi paths. If one is under SCRIPT_DIR but does not exist here,
    try the same relative path under this repo's StartHere folder."""
    if p is None or os.path.exists(p):
        return p
    script_dir = sweep["variables"].get("SCRIPT_DIR")
    if script_dir and os.path.abspath(p).startswith(os.path.abspath(script_dir) + os.sep):
        alt = os.path.join(STARTHERE_DIR, os.path.relpath(p, script_dir))
        if os.path.exists(alt):
            return alt
    return p

CONFIG_FILE = CONFIG_OVERRIDE or _local_fallback(sweep["config_file"])
OUTDIR      = OUTDIR_OVERRIDE or sweep["outdir"]
LOGDIR      = LOGDIR_OVERRIDE or sweep["logdir"]
STATUS_DIR  = os.path.join(LOGDIR, SWEEP_NAME)                      # <LOGDIR>/<name>/
STATUS_CSV  = os.path.join(STATUS_DIR, f"{SWEEP_NAME}_status.csv")  # <name>_status.csv

with open(CONFIG_FILE) as f:
    BASE_CONFIG = json.load(f)

# ---- 2. The grid: {flag: [typed values]} ---------------------------------------------
FLAG_MAP = cli_flag_map(RUN_MODEL)
grid_str = SWEEP_GRID_MANUAL if SWEEP_GRID_MANUAL is not None else sweep["grid"]
unknown = [fl for fl in grid_str if fl not in FLAG_MAP]
if unknown:
    raise KeyError(f"Swept flag(s) {unknown} not found as config overrides in {RUN_MODEL}. "
                   f"Known flags: {sorted(FLAG_MAP)}")
# cast with the flag's argparse type, exactly as run_model_Hof.py receives it
# (e.g. "0" -> 0.0, so the filename token is "psi0.0" like the real file)
GRID = {fl: [FLAG_MAP[fl][2](v) for v in vals] for fl, vals in grid_str.items()}
SWEEP_PARAMS = list(GRID)        # column order = launch-argument order

print(f"Sweep script : {SWEEP_SCRIPT_PATH}")
print(f"Config       : {CONFIG_FILE}")
print(f"Output dir   : {OUTDIR}")
print(f"Status CSV   : {STATUS_CSV}")
print(f"Grid ({' x '.join(str(len(v)) for v in GRID.values())} = "
      f"{int(np.prod([len(v) for v in GRID.values()]))} runs):")
for fl, vals in GRID.items():
    sec, key, _ = FLAG_MAP[fl]
    print(f"  --{fl:<10s} -> {sec}.{key:<10s} {vals}")

# ---- 3. One row per combination ------------------------------------------------------
rows = []
for combo in itertools.product(*GRID.values()):
    params = dict(zip(SWEEP_PARAMS, combo))
    cfg = config_for_run(BASE_CONFIG, params, FLAG_MAP)
    fname = expected_filename(cfg)
    status, runtime_hr = run_status(os.path.join(OUTDIR, fname), expected_t_end_Myr(cfg))
    rows.append({"filename": fname, **params, "status": status, "runtime_hr": runtime_hr})

status_df = pd.DataFrame(rows, columns=["filename", *SWEEP_PARAMS, "status", "runtime_hr"])

# Sanity check: .h5 files in OUTDIR that the grid did NOT predict usually mean the
# filename format or the grid parsing is off (or leftovers from another sweep).
if os.path.isdir(OUTDIR):
    unexpected = sorted(set(f for f in os.listdir(OUTDIR) if f.endswith(".h5")) - set(status_df["filename"]))
    if unexpected:
        print(f"\nWARNING: {len(unexpected)} .h5 file(s) in OUTDIR not predicted by the grid, e.g.:")
        for f in unexpected[:5]:
            print("   ", f)
else:
    print(f"\nWARNING: OUTDIR does not exist here ({OUTDIR}) -- every run will show as 'missing'. "
          f"Set OUTDIR_OVERRIDE if the files are elsewhere.")

# ---- 4. Save and summarise -----------------------------------------------------------
os.makedirs(STATUS_DIR, exist_ok=True)
status_df.to_csv(STATUS_CSV, index=False)
print(f"\nWrote {len(status_df)} rows -> {STATUS_CSV}\n")
print(status_df["status"].value_counts().to_string())
rt = status_df.loc[status_df["status"] == "complete", "runtime_hr"]
if rt.notna().any():
    print(f"\nRuntime of complete runs [hours]: median {rt.median():.2f}, max {rt.max():.2f}")
status_df

Sweep script : /project/6000781/imontesd/DiscEvolution/example/StartHere/run_popsynth1.00_hof.sh
Config       : /home/imontesd/projects/def-mbalogh/imontesd/DiscEvolution/example/StartHere/config/popsynth/DiscConfig_Hof_popsynth1.00.json
Output dir   : /project/def-mbalogh/imontesd/output/DiscEvolution/popsynth/1.00
Status CSV   : /project/def-mbalogh/imontesd/output/DiscEvolution/logs/run_popsynth1.00_hof/run_popsynth1.00_hof_status.csv
Grid (4 x 3 x 3 x 3 x 3 = 324 runs):
  --psi_DW     -> winds.psi_DW     [0.0, 10.0, 100.0, 1000.0]
  --alpha_SS   -> calibration.alpha_SS   [0.0001, 0.001, 0.01]
  --M          -> disc.M          [0.001, 0.01, 0.1]
  --Rd         -> disc.Rd         [25.0, 50.0, 100.0]
  --e_rad      -> winds.e_rad      [0.8, 0.9, 1.0]

Wrote 324 rows -> /project/def-mbalogh/imontesd/output/DiscEvolution/logs/run_popsynth1.00_hof/run_popsynth1.00_hof_status.csv

status
complete             201
Mdot_out_of_range    114
crashed                9

Runtime of complete runs [

,filename,psi_DW,alpha_SS,M,Rd,e_rad,status,runtime_hr
0,hof_discwind_popsynth1.00_gamma1_psi0.0_erad0.8_aSS1.0e-04_Mdot1.0e-08_M1.0e-03_Rd2.5e+01.h5,0.0,0.0001,0.001,25.0,0.8,Mdot_out_of_range,0.000278
1,hof_discwind_popsynth1.00_gamma1_psi0.0_erad0.9_aSS1.0e-04_Mdot1.0e-08_M1.0e-03_Rd2.5e+01.h5,0.0,0.0001,0.001,25.0,0.9,Mdot_out_of_range,0.000556
2,hof_discwind_popsynth1.00_gamma1_psi0.0_erad1.0_aSS1.0e-04_Mdot1.0e-08_M1.0e-03_Rd2.5e+01.h5,0.0,0.0001,0.001,25.0,1.0,Mdot_out_of_range,0.000833
3,hof_discwind_popsynth1.00_gamma1_psi0.0_erad0.8_aSS1.0e-04_Mdot1.0e-08_M1.0e-03_Rd5.0e+01.h5,0.0,0.0001,0.001,50.0,0.8,Mdot_out_of_range,0.000278
4,hof_discwind_popsynth1.00_gamma1_psi0.0_erad0.9_aSS1.0e-04_Mdot1.0e-08_M1.0e-03_Rd5.0e+01.h5,0.0,0.0001,0.001,50.0,0.9,Mdot_out_of_range,0.000556
5,hof_discwind_popsynth1.00_gamma1_psi0.0_erad1.0_aSS1.0e-04_Mdot1.0e-08_M1.0e-03_Rd5.0e+01.h5,0.0,0.0001,0.001,50.0,1.0,Mdot_out_of_range,0.000278
6,hof_discwind_popsynth1.00_gamma1_psi0.0_erad0.8_aSS1.0e-04_Mdot1.0e-08_M1.0e-03_Rd1.0e+02.h5,0.0,0.0001,0.001,100.0,0.8,Mdot_out_of_range,0.000278
7,hof_discwind_popsynth1.00_gamma1_psi0.0_erad0.9_aSS1.0e-04_Mdot1.0e-08_M1.0e-03_Rd1.0e+02.h5,0.0,0.0001,0.001,100.0,0.9,Mdot_out_of_range,0.000833
8,hof_discwind_popsynth1.00_gamma1_psi0.0_erad1.0_aSS1.0e-04_Mdot1.0e-08_M1.0e-03_Rd1.0e+02.h5,0.0,0.0001,0.001,100.0,1.0,Mdot_out_of_range,0.000556
9,hof_discwind_popsynth1.00_gamma1_psi0.0_erad0.8_aSS1.0e-04_Mdot1.0e-08_M1.0e-02_Rd2.5e+01.h5,0.0,0.0001,0.010,25.0,0.8,complete,0.458889
